# Cookbook: Company Research Workflow

A step-by-step guide to researching a public company using `python-sec`.
This notebook covers ticker resolution, company metadata, recent filings,
XBRL financial data, and exporting results to pandas DataFrames.

**What you'll learn:**
1. Look up any company by ticker or CIK
2. Retrieve company metadata (SIC, exchange, fiscal year)
3. Browse recent filings and filter by form type
4. Access XBRL financial facts
5. Export everything to DataFrames for further analysis

## 1. Setup

In [1]:
from edgar.client import EdgarClient

# SEC EDGAR requires a User-Agent header in the format "Name email@example.com".
# Replace with your own identity before running against the live API.
edgar_client = EdgarClient(user_agent="Your Name your-email@example.com")

## 2. Find a Company by Ticker

Use `company()` to look up any company by its stock ticker symbol.
The library resolves the ticker to a CIK using SEC's company tickers file.

In [2]:
apple = edgar_client.company("AAPL")

print(f"Company: {apple.name}")
print(f"CIK:     {apple.cik}")
print(f"Ticker:  {apple.ticker}")

Company: Apple Inc.
CIK:     0000320193
Ticker:  AAPL


## 3. Company Metadata

`get_info()` returns a `CompanyInfo` model with SIC code, exchange, fiscal year,
and a list of recent filings. In Jupyter, it auto-renders as an HTML table.

In [3]:
info = apple.get_info()
info  # Auto-renders as an HTML card in Jupyter

<CompanyInfo name='Apple Inc.' cik='0000320193' tickers='AAPL'>

In [ ]:
# Access individual metadata properties.
print(f"SIC Code:        {info.sic} — {info.sic_description}")
print(f"Fiscal Year End: {info.fiscal_year_end}")
print(f"Tickers:         {info.tickers}")
print(f"Exchanges:       {info.exchanges}")

## 4. Recent Submissions

The `recent_submissions` property returns `Submission` model objects — one for each
recent filing in the company's history.

In [4]:
# Show the 5 most recent submissions.
for sub in info.recent_submissions[:5]:
    print(f"{sub.filing_date}  {sub.form:<10}  {sub.primary_doc_description}")

2026-04-17  4           FORM 4
2026-04-17  4           FORM 4
2026-04-03  4           FORM 4
2026-04-03  4           FORM 4
2026-04-03  4           FORM 4


## 5. Filings Filtered by Form Type

`get_filings(form="10-K")` returns only annual reports as `Filing` model objects.

In [5]:
ten_k_filings = apple.get_filings(form="10-K")

print(f"Found {len(ten_k_filings)} 10-K filings.\n")
for f in ten_k_filings[:3]:
    print(f"{f.filing_date}  {f.form_type}  {f.accession_number}")

Found 34 10-K filings.

2025-10-31T06:01:26-04:00  10-K  0000320193-25-000079
2024-11-01T06:01:36-04:00  10-K  0000320193-24-000123
2023-11-02T18:08:27-04:00  10-K  0000320193-23-000106


In [ ]:
# Display the most recent 10-K as an HTML card.
ten_k_filings[0]

## 6. XBRL Financial Facts

`get_facts()` returns all XBRL data the company has filed. You can then query
specific concepts like Revenue, Assets, or NetIncomeLoss.

In [6]:
facts = apple.get_facts()

print(f"Entity:     {facts.entity_name}")
print(f"Taxonomies: {facts.taxonomies}")
print(f"us-gaap concepts: {len(facts.concepts('us-gaap'))}")

Entity:     Apple Inc.
Taxonomies: ['dei', 'us-gaap']
us-gaap concepts: 503


In [7]:
# Get the last 5 annual revenue data points.
revenue = facts.get("us-gaap", "Revenues", unit="USD")
for r in revenue[-5:]:
    print(f"FY{r.fiscal_year} {r.fiscal_period}  ${r.value:>15,.0f}  (filed {r.filed})")

FY2018 FY  $ 88,293,000,000  (filed 2018-11-05)
FY2018 FY  $ 61,137,000,000  (filed 2018-11-05)
FY2018 FY  $ 53,265,000,000  (filed 2018-11-05)
FY2018 FY  $265,595,000,000  (filed 2018-11-05)
FY2018 FY  $ 62,900,000,000  (filed 2018-11-05)


## 7. Export to DataFrames

Convert any list of model objects to a pandas DataFrame with `to_dataframe()`.

In [8]:
from edgar.models import to_dataframe

# Filings as a DataFrame.
filings_df = to_dataframe(ten_k_filings)
filings_df.head()

,accession_number,filing_date,form_type,summary,title,url
0,0000320193-25-000079,2025-10-31T06:01:26-04:00,10-K,<b>Filed:</b> 2025-10-31 <b>AccNo:</b> 0000320...,"10-K - Annual report [Section 13 and 15(d), n...",https://www.sec.gov/Archives/edgar/data/320193...
1,0000320193-24-000123,2024-11-01T06:01:36-04:00,10-K,<b>Filed:</b> 2024-11-01 <b>AccNo:</b> 0000320...,"10-K - Annual report [Section 13 and 15(d), n...",https://www.sec.gov/Archives/edgar/data/320193...
2,0000320193-23-000106,2023-11-02T18:08:27-04:00,10-K,<b>Filed:</b> 2023-11-03 <b>AccNo:</b> 0000320...,"10-K - Annual report [Section 13 and 15(d), n...",https://www.sec.gov/Archives/edgar/data/320193...
3,0000320193-22-000108,2022-10-27T18:01:14-04:00,10-K,<b>Filed:</b> 2022-10-28 <b>AccNo:</b> 0000320...,"10-K - Annual report [Section 13 and 15(d), n...",https://www.sec.gov/Archives/edgar/data/320193...
4,0000320193-21-000105,2021-10-28T18:04:28-04:00,10-K,<b>Filed:</b> 2021-10-29 <b>AccNo:</b> 0000320...,"10-K - Annual report [Section 13 and 15(d), n...",https://www.sec.gov/Archives/edgar/data/320193...


In [9]:
# Revenue facts as a DataFrame — ready for time-series analysis.
revenue_df = facts.to_dataframe("us-gaap", "Revenues", unit="USD")
revenue_df.tail(10)

,accession_number,end,filed,fiscal_period,fiscal_year,form,frame,start,value
1,0000320193-18-000145,2016-12-31,2018-11-05,FY,2018,10-K,CY2016Q4,2016-09-25,78351000000
2,0000320193-18-000145,2017-04-01,2018-11-05,FY,2018,10-K,CY2017Q1,2017-01-01,52896000000
3,0000320193-18-000145,2017-07-01,2018-11-05,FY,2018,10-K,CY2017Q2,2017-04-02,45408000000
4,0000320193-18-000145,2017-09-30,2018-11-05,FY,2018,10-K,CY2017,2016-09-25,229234000000
5,0000320193-18-000145,2017-09-30,2018-11-05,FY,2018,10-K,CY2017Q3,2017-07-02,52579000000
6,0000320193-18-000145,2017-12-30,2018-11-05,FY,2018,10-K,CY2017Q4,2017-10-01,88293000000
7,0000320193-18-000145,2018-03-31,2018-11-05,FY,2018,10-K,CY2018Q1,2017-12-31,61137000000
8,0000320193-18-000145,2018-06-30,2018-11-05,FY,2018,10-K,CY2018Q2,2018-04-01,53265000000
9,0000320193-18-000145,2018-09-29,2018-11-05,FY,2018,10-K,CY2018,2017-10-01,265595000000
10,0000320193-18-000145,2018-09-29,2018-11-05,FY,2018,10-K,CY2018Q3,2018-07-01,62900000000


## 8. Compare Multiple Companies

The same workflow works for any public company. Let's compare Apple and Microsoft.

In [10]:
msft = edgar_client.company("MSFT")
msft_facts = msft.get_facts()

msft_revenue = msft_facts.get("us-gaap", "Revenues", unit="USD")

print("Apple Revenue (last 3 annual):")
for r in revenue[-3:]:
    print(f"  FY{r.fiscal_year} {r.fiscal_period}: ${r.value:>15,.0f}")

print("\nMicrosoft Revenue (last 3 annual):")
for r in msft_revenue[-3:]:
    print(f"  FY{r.fiscal_year} {r.fiscal_period}: ${r.value:>15,.0f}")

Apple Revenue (last 3 annual):
  FY2018 FY: $ 53,265,000,000
  FY2018 FY: $265,595,000,000
  FY2018 FY: $ 62,900,000,000

Microsoft Revenue (last 3 annual):
  FY2011 Q1: $ 16,195,000,000
  FY2011 Q2: $ 36,148,000,000
  FY2011 Q2: $ 19,953,000,000


## Summary

| Step | Method | Returns |
|------|--------|---------|
| Look up company | `client.company("AAPL")` | `Company` |
| Company metadata | `company.get_info()` | `CompanyInfo` |
| Recent filings | `company.get_filings(form="10-K")` | `list[Filing]` |
| XBRL facts | `company.get_facts()` | `Facts` |
| Query a concept | `facts.get("us-gaap", "Revenues")` | `list[Fact]` |
| Export to pandas | `to_dataframe(filings)` | `DataFrame` |

All response objects auto-render as HTML tables in Jupyter via `_repr_html_()`.